# 直接用AMT神经网络的输出进行节拍跟踪

In [ ]:
import numpy as np
import soundfile as sf

# file_name = "加藤達也-Always in my heart"
file_name = "彩音 - いつもこの場所で"

waveform, sr = sf.read(file_name+'.wav')
if sr != 44100:
    raise ValueError("采样率必须为 44100 Hz")
if waveform.ndim > 1:
    waveform = waveform.mean(axis=1)

print(waveform.shape, sr)
onset = np.load(file_name+'_onset.npy')
print(onset.shape)
onset_sr = 44100 / 512  # onset的采样率

onset_sum = np.sum(onset, axis=0)
onset_sum = onset_sum / (np.max(onset_sum) + 1e-5)

def moving_max_normalize(arr, window_size):
    """
    对输入数组进行滑动最大值归一化。
    每个点除以其窗口内的最大值。
    """
    half = window_size // 2
    normed = np.zeros_like(arr)
    for i in range(len(arr)):
        left = max(0, i - half)
        right = min(len(arr), i + half + 1)
        normed[i] = arr[i] / (np.max(arr[left:right]) + 1e-5)
    return normed

from scipy.signal import butter, lfilter

def iir_highpass_filter(arr, cutoff, fs, order=4):
    """
    用IIR高通滤波器对数组进行滤波。
    arr: 输入信号
    cutoff: 截止频率（Hz）
    fs: 采样率（Hz）
    order: 滤波器阶数
    返回滤波后的信号
    """
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    filtered = lfilter(b, a, arr)
    return filtered

In [ ]:
from matplotlib import pyplot as plt

# 只保留前20秒
duration = 20
num_samples = int(duration * onset_sr)
print("num_samples:", num_samples)
t = np.arange(num_samples) / onset_sr

plt.figure(figsize=(12, 4))
plt.plot(t, onset_sum[:num_samples])
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('onset_sum (First 20 seconds)')
plt.show()

In [ ]:
from madmom.features.beats import DBNBeatTrackingProcessor

beat_processor = DBNBeatTrackingProcessor(
    min_bpm=40,
    max_bpm=200,
    # 速度越恒定，值越大
    transition_lambda=1000,
    fps=onset_sr,
    online=False,
    threshold=1e-8,
    num_tempi=64
)

# 执行节拍跟踪 → 输出单位为秒
beat_times = beat_processor(moving_max_normalize(onset_sum, window_size=int(onset_sr * 3)))

print("检测到的节拍时间（秒）:")
print(beat_times)

In [ ]:
from IPython.display import Audio

# 读取cowbell音频
cowbell, cowbell_sr = sf.read('cowbell.wav')
if cowbell.ndim > 1:
    cowbell = cowbell.mean(axis=1)
cowbell /= np.abs(cowbell).max()

# 生成节拍数组（与waveform长度一致），用beat_times指导
beat_audio = np.zeros_like(waveform)
for t in beat_times:
    idx = int(t * sr)  # 将秒转换为采样点
    end_idx = min(idx + len(cowbell), len(beat_audio))
    beat_audio[idx:end_idx] += cowbell[:end_idx-idx]

beat_audio /= np.max(np.abs(beat_audio))  # 防止溢出

# 合成原音频和节拍
mix_audio = waveform + beat_audio
mix_audio /= np.abs(mix_audio).max()

# 使用IPython播放混合音频
Audio(mix_audio, rate=sr)